In [ ]:
from utils import ASSETS_DIR
import torch
import torch.nn.functional as F
from models.quantum.quantumMap import quantumAnsatz, VQC
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import torch.nn as nn
from utils.models import load_all_skops
import torch.optim as optim
from sklearn.utils import resample
from qiskit.circuit.library import real_amplitudes
from qiskit.circuit import Parameter, QuantumCircuit, ParameterVector
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_aer.primitives import EstimatorV2
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms.optimizers import SPSA
import time
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from utils.dataset import undersample_dataset





n_qubits = 6
dim = [4, 8, 16, 32]

objs = load_all_skops(ASSETS_DIR / "seeds_11")
objs.pop("scaler")


pca32 = objs['pca_32']

dataset = pd.read_csv(ASSETS_DIR / "seeds_11/train_11.csv")
dataset = undersample_dataset(dataset, label_col='label', random_seed=11)
dataset = resample(dataset, replace=False, n_samples=4000, stratify=dataset["label"], random_state=11)
val = pd.read_csv(ASSETS_DIR / "seeds_11/val_11.csv")
val = undersample_dataset(val, label_col='label', random_seed=11)
X_train = dataset.drop(columns=["label"])
X_train = pca32.transform(X_train.values)
Y_train = dataset["label"]
X_val = val.drop(columns=["label"])
X_val = pca32.transform(X_val.values)
Y_val = val["label"]

del dataset

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

tensor = torch.from_numpy(X_train).float()
tensor.size()
tensor_Y = torch.from_numpy(Y_train.values).long()

tensor_val = torch.from_numpy(X_val).float()
tensor_Y_val = torch.from_numpy(Y_val.values).long()

Caricamento di: scaler.skops...
scaler caricato con successo.
Caricamento di: pca_16.skops...
pca_16 caricato con successo.
Caricamento di: pca_4.skops...
pca_4 caricato con successo.
Caricamento di: pca_32.skops...
pca_32 caricato con successo.
Caricamento di: pca_8.skops...
pca_8 caricato con successo.
Dimensione classe minoritaria trovata: 7093 elementi.
Dimensione classe minoritaria trovata: 886 elementi.


(np.float64(0.0), np.float64(6.283185307179587))

In [2]:
def zero_padding(n_qubits, n_dim):
    resto = n_dim % n_qubits
    if resto == 0:
        pad_size = 0
    else:
        pad_size = n_qubits - resto  # Nel caso 16 % 6 -> resto 4 -> servono 2 zeri

    d_padded = n_dim + pad_size          

    print(f"Dimensione originale: {n_dim}, Zeri aggiunti: {pad_size}, Dimensione padded: {d_padded}")
    return d_padded


padding = zero_padding(n_qubits, dim[3])

Dimensione originale: 32, Zeri aggiunti: 4, Dimensione padded: 36


In [15]:
ansatz = real_amplitudes(num_qubits=6, reps=2)

input_layer = QuantumCircuit(n_qubits)
input_params = ParameterVector("x", length=padding)
for i in range(padding):
    input_layer.ry(input_params[i], i % n_qubits)  # Applica la rotazione Ry al qubit i%n_qubits

full_ansatz = input_layer.compose(ansatz)


In [7]:
from qiskit.circuit.library import ZZFeatureMap
from qiskit.circuit import ParameterVector, QuantumCircuit



n_qubits = 6
# padding è 36 dal tuo calcolo precedente
input_params = ParameterVector("x", padding)
input_layer = QuantumCircuit(n_qubits)

ansatz = real_amplitudes(num_qubits=6, reps=2)

# Cicliamo a passi di 6 (n_qubits) attraverso le 36 features
for i in range(0, padding, n_qubits):
    # Estraiamo la porzione di feature (6 alla volta)
    chunk_params = input_params[i : i + n_qubits]
    
    # Creiamo un "mattoncino" ZZFeatureMap per questi 6 qubit (reps=1 è sufficiente)
    # entanglement='linear' o 'circular' riduce la profondità rispetto a 'full'
    fm_chunk = ZZFeatureMap(feature_dimension=n_qubits, reps=1, entanglement='linear')
    
    # Leghiamo i parametri del blocco corrente e componiamo il circuito
    fm_bound = fm_chunk.assign_parameters(chunk_params)
    input_layer = input_layer.compose(fm_bound)

# Aggiungi il tuo ansatz parametrizzato alla fine come hai già fatto
full_ansatz = input_layer.compose(ansatz).decompose()

In [16]:



weight_params = [param for param in full_ansatz.parameters if param.name.startswith("θ")]

observables = []
for i in range(n_qubits):
    # Mette una 'Z' sul qubit i, e l'identità 'I' su tutti gli altri
    obs = SparsePauliOp.from_sparse_list([("Z", [i], 1.0)], num_qubits=n_qubits)
    observables.append(obs)


#Estimator = EstimatorV2(options={"run_options": {"device": "GPU"}})

estimator = AerEstimator()
estimator.options.simulator = {
    "method": "statevector",  
    "device": "GPU",          # Usa la scheda video
    "cuStateVec_enable": True # Attiva i driver quantistici NVIDIA ad alte prestazioni
}

qnn = EstimatorQNN(
    circuit=full_ansatz,
    observables=observables,
    input_params=input_params,
    weight_params=weight_params,
    estimator=estimator
)

print(
    f"Number of input features for EstimatorQNN: {qnn.num_inputs} \nInput: {qnn.input_params}"
)
print(
    f"Number of trainable weights for EstimatorQNN: {qnn.num_weights} \nWeights: {qnn.weight_params}"
)

X_padded = F.pad(tensor, (0, padding - dim[3]), mode="constant", value=0.0)
dataset_train = torch.utils.data.TensorDataset(
    X_padded.float(),
    torch.from_numpy(Y_train.values).long()
)
print(f"Dataset train size: {dataset_train.__len__()} samples")
dataloader_train = torch.utils.data.DataLoader(dataset_train, batch_size=128, shuffle=True, num_workers=12, pin_memory=True)

X_padded_val = F.pad(tensor_val, (0, padding - dim[3]), mode="constant", value=0.0)
dataset_val = torch.utils.data.TensorDataset(
    X_padded_val.float(),
    torch.from_numpy(Y_val.values).long()
)
print(f"Dataset val size: {dataset_val.__len__()} samples")
dataloader_val = torch.utils.data.DataLoader(dataset_val, batch_size=128, shuffle=False, num_workers=12, pin_memory=True)




No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


Number of input features for EstimatorQNN: 36 
Input: [ParameterVectorElement(x[0]), ParameterVectorElement(x[1]), ParameterVectorElement(x[2]), ParameterVectorElement(x[3]), ParameterVectorElement(x[4]), ParameterVectorElement(x[5]), ParameterVectorElement(x[6]), ParameterVectorElement(x[7]), ParameterVectorElement(x[8]), ParameterVectorElement(x[9]), ParameterVectorElement(x[10]), ParameterVectorElement(x[11]), ParameterVectorElement(x[12]), ParameterVectorElement(x[13]), ParameterVectorElement(x[14]), ParameterVectorElement(x[15]), ParameterVectorElement(x[16]), ParameterVectorElement(x[17]), ParameterVectorElement(x[18]), ParameterVectorElement(x[19]), ParameterVectorElement(x[20]), ParameterVectorElement(x[21]), ParameterVectorElement(x[22]), ParameterVectorElement(x[23]), ParameterVectorElement(x[24]), ParameterVectorElement(x[25]), ParameterVectorElement(x[26]), ParameterVectorElement(x[27]), ParameterVectorElement(x[28]), ParameterVectorElement(x[29]), ParameterVectorElement(x[

In [17]:
spsa = SPSA(maxiter=20, learning_rate=0.05, perturbation=0.2)

class_layer = nn.Linear(n_qubits, 4) 
optimizer_class = optim.Adam(class_layer.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

q_weights = np.random.uniform(0, 2 * np.pi, qnn.num_weights)
epochs = 10

def spsa_objective(weights):
            # 1. Esecuzione del circuito sui dati del batch
            q_out = qnn.forward(X_np, weights) # shape: (batch_size, 6)
            
            # 2. Passaggio nel layer classico solo per misurare l'errore
            with torch.no_grad(): 
                q_tensor = torch.tensor(q_out, dtype=torch.float32)
                logits = class_layer(q_tensor)
                loss = loss_fn(logits, Y_batch)
                
            return loss.item()

# Assumiamo di avere il dataloader_train impostato con batch_size tra 64 e 128
for epoch in range(epochs):
    print(f"\n--- Inizio Epoca {epoch+1}/{epochs} ---")
    
    time_epoch_start = time.time()
    for batch_idx, (X_batch, Y_batch) in enumerate(dataloader_train): # Sostituisci con il tuo loader di training
        torch.cuda.empty_cache()  # Pulisce la cache GPU per evitare out-of-memory
        t_bach_start = time.time()

        # Qiskit richiede array NumPy
        X_np = X_batch.numpy()
        
        # ==========================================
        # FASE 1: Addestramento Quantistico (SPSA)
        # ==========================================
        class_layer.eval() # Congela i pesi classici
        
        
        
        # SPSA stima il gradiente e aggiorna ESCLUSIVAMENTE i pesi quantistici
        res = spsa.minimize(fun=spsa_objective, x0=q_weights)
        q_weights = res.x 
        
        # ==========================================
        # FASE 2: Addestramento Classico (PyTorch Adam)
        # ==========================================
        class_layer.train() # Riattiva il tracciamento dei gradienti
        optimizer_class.zero_grad()
        
        # Rieseguiamo il forward quantistico con i pesi aggiornati per avere dati allineati
        q_out_updated = qnn.forward(X_np, q_weights)
        q_tensor_updated = torch.tensor(q_out_updated, dtype=torch.float32)
        
        # Calcolo classico puro e backpropagation
        logits = class_layer(q_tensor_updated)
        loss = loss_fn(logits, Y_batch)
        
        loss.backward()
        optimizer_class.step() # Aggiorna ESCLUSIVAMENTE i pesi di nn.Linear
        
        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx} completato - Loss: {loss.item():.4f}")
            t_batch_end = time.time()
            print(f"  Tempo per Batch {batch_idx}: {t_batch_end - t_bach_start:.2f} secondi")
        
    time_epoch_end = time.time()
    print(f"Tempo per Epoca {epoch+1}: {time_epoch_end - time_epoch_start:.2f} secondi")
    print(f"Fine Epoca {epoch+1} completata.")


--- Inizio Epoca 1/10 ---
  Batch 0 completato - Loss: 1.4784
  Tempo per Batch 0: 4.77 secondi
  Batch 10 completato - Loss: 1.3859
  Tempo per Batch 10: 4.53 secondi
  Batch 20 completato - Loss: 1.3715
  Tempo per Batch 20: 4.52 secondi
  Batch 30 completato - Loss: 1.3434
  Tempo per Batch 30: 4.51 secondi
Tempo per Epoca 1: 140.34 secondi
Fine Epoca 1 completata.

--- Inizio Epoca 2/10 ---
  Batch 0 completato - Loss: 1.3896
  Tempo per Batch 0: 4.44 secondi
  Batch 10 completato - Loss: 1.3722
  Tempo per Batch 10: 4.48 secondi
  Batch 20 completato - Loss: 1.3699
  Tempo per Batch 20: 4.65 secondi
  Batch 30 completato - Loss: 1.3481
  Tempo per Batch 30: 4.40 secondi
Tempo per Epoca 2: 140.21 secondi
Fine Epoca 2 completata.

--- Inizio Epoca 3/10 ---
  Batch 0 completato - Loss: 1.3402
  Tempo per Batch 0: 4.62 secondi
  Batch 10 completato - Loss: 1.3262
  Tempo per Batch 10: 4.37 secondi
  Batch 20 completato - Loss: 1.3163
  Tempo per Batch 20: 4.40 secondi
  Batch 30 comp

In [4]:
from qiskit_algorithms.optimizers import QNSPSA
from qiskit.primitives import StatevectorSampler
# ========== IMPROVED TRAINING WITH METRICS ==========

sampler = StatevectorSampler()
fidelity = QNSPSA.get_fidelity(ansatz, sampler)

qnspsa = QNSPSA(fidelity, maxiter=100)


class_layer = nn.Linear(n_qubits, 4) 
optimizer_class = optim.Adam(class_layer.parameters(), lr=0.0005)  # Slightly lower LR
loss_fn = nn.CrossEntropyLoss()

q_weights = np.random.uniform(0, 2 * np.pi, qnn.num_weights)
epochs = 30  # Increased from 10

# ===== METRIC TRACKING =====
train_losses = []
val_losses = []
val_accuracies = []

def compute_validation_metrics():
    """Compute validation loss and accuracy"""
    class_layer.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, Y_batch in dataloader_val:
            X_np = X_batch.numpy()
            q_out = qnn.forward(X_np, q_weights)
            q_tensor = torch.tensor(q_out, dtype=torch.float32)
            logits = class_layer(q_tensor)
            loss = loss_fn(logits, Y_batch)
            
            total_loss += loss.item()
            _, preds = torch.max(logits, dim=1)
            correct += (preds == Y_batch).sum().item()
            total += Y_batch.size(0)
    
    avg_loss = total_loss / len(dataloader_val)
    accuracy = correct / total if total > 0 else 0.0
    return avg_loss, accuracy

def qnspsa_objective(x):
    return estimator.run([(ansatz, observables, x)]).result()[0].data.evs

def spsa_objective(weights):
    """SPSA objective function with batched evaluation"""
    q_out = qnn.forward(X_np, weights)
    with torch.no_grad(): 
        q_tensor = torch.tensor(q_out, dtype=torch.float32)
        logits = class_layer(q_tensor)
        loss = loss_fn(logits, Y_batch)
    return loss.item()

# ===== MAIN TRAINING LOOP =====
for epoch in range(epochs):
    print(f"\n{'='*60}")
    print(f"Epoca {epoch+1}/{epochs}")
    print(f"{'='*60}")
    
    time_epoch_start = time.time()
    epoch_losses = []
    batch_count = 0
    
    for batch_idx, (X_batch, Y_batch) in enumerate(dataloader_train):
        torch.cuda.empty_cache()
        
        X_np = X_batch.numpy()
        
        # =========== PHASE 1: Quantum Training (SPSA) ===========
        class_layer.eval()  # Freeze classical weights
        
        # SPSA minimization
        res = qnspsa.minimize(fun=spsa_objective, x0=q_weights)
        q_weights = res.x 
        
        # =========== PHASE 2: Classical Training (Adam) ===========
        class_layer.train()
        optimizer_class.zero_grad()
        
        q_out_updated = qnn.forward(X_np, q_weights)
        q_tensor_updated = torch.tensor(q_out_updated, dtype=torch.float32)
        logits = class_layer(q_tensor_updated)
        loss = loss_fn(logits, Y_batch)
        
        loss.backward()
        optimizer_class.step()
        
        epoch_losses.append(loss.item())
        batch_count += 1
        
        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx:3d} | Loss: {loss.item():.4f}")
    
    # =========== EPOCH STATISTICS ===========
    avg_epoch_loss = np.mean(epoch_losses)
    train_losses.append(avg_epoch_loss)
    
    # Validate every 5 epochs
    if (epoch + 1) % 5 == 0 or (epoch + 1) == epochs:
        val_loss, val_acc = compute_validation_metrics()
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        print(f"\n  📊 Validation Results:")
        print(f"     Validation Loss: {val_loss:.4f}")
        print(f"     Validation Accuracy: {val_acc*100:.2f}%")
    
    time_epoch_end = time.time()
    epoch_time = time_epoch_end - time_epoch_start
    print(f"\n  ⏱️  Epoch Time: {epoch_time:.2f}s")
    print(f"  📈 Avg Training Loss: {avg_epoch_loss:.4f}")



Epoca 1/30
  Batch   0 | Loss: 1.3790
  Batch  10 | Loss: 1.3993


KeyboardInterrupt: 

In [ ]:
# --- 4. Valutazione sul Validation Set e Confusion Matrix ---

print("\n--- Valutazione sul Validation Set ---")
class_layer.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, Y_batch in dataloader_val:
        X_np = X_batch.numpy()
        
        # Forward quantistico
        q_out = qnn.forward(X_np, q_weights)
        
        # Forward classico (applicando lo stesso moltiplicatore del training)
        q_tensor = torch.tensor(q_out, dtype=torch.float32) * 2.0
        logits = class_layer(q_tensor)
        
        # Estrazione delle classi predette
        _, preds = torch.max(logits, dim=1)
        
        all_preds.extend(preds.numpy())
        all_labels.extend(Y_batch.numpy())

# Generazione e stampa della matrice
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1, 2, 3])

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax, values_format='d')
plt.title("Confusion Matrix del Modello Ibrido Quantum-Classico")
plt.xlabel("Classe Predetta")
plt.ylabel("Classe Reale")
plt.show()

In [ ]:

# ===== PLOT TRAINING METRICS =====

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Training Loss
axes[0].plot(range(1, len(train_losses) + 1), train_losses, marker='o', label='Training Loss', color='blue')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss Curve', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Plot 2: Validation Accuracy
if val_accuracies:
    val_epochs = [5*i for i in range(1, len(val_accuracies)+1)]
    axes[1].plot(val_epochs, [acc*100 for acc in val_accuracies], marker='s', label='Validation Accuracy', color='green', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].set_title('Validation Accuracy', fontsize=14, fontweight='bold')
    axes[1].set_ylim([0, 105])
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

plt.tight_layout()
plt.savefig('training_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("📈 TRAINING SUMMARY")
print("="*60)
print(f"Final Training Loss: {train_losses[-1]:.4f}")
if val_accuracies:
    print(f"Final Validation Accuracy: {val_accuracies[-1]*100:.2f}%")
    print(f"Best Validation Accuracy: {max(val_accuracies)*100:.2f}%")
print("="*60)


In [ ]:
Y_val.info()
Y_val.value_counts()